# Falcon-Perception — MLX Demo

Open-vocabulary **object detection** and **instance segmentation** on Apple Silicon.

| Model | Params | Detection | Segmentation |
|-------|--------|-----------|-------------|
| [`tiiuae/Falcon-Perception`](https://huggingface.co/tiiuae/Falcon-Perception) | ~0.6B | Yes | Yes |
| [`tiiuae/Falcon-Perception-300M`](https://huggingface.co/tiiuae/Falcon-Perception-300M) | ~0.3B | Yes | No |

## Setup

In [ ]:
!pip install -U mlx-vlm matplotlib

In [ ]:
import numpy as np
import mlx.core as mx
import requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

%matplotlib inline

COLORS = [
    (0.12, 0.47, 0.71),  # blue
    (1.00, 0.50, 0.05),  # orange
    (0.17, 0.63, 0.17),  # green
    (0.84, 0.15, 0.16),  # red
    (0.58, 0.40, 0.74),  # purple
    (0.55, 0.34, 0.29),  # brown
    (0.89, 0.47, 0.76),  # pink
    (0.09, 0.75, 0.81),  # cyan
]


def fetch_image(url):
    return Image.open(BytesIO(requests.get(url, timeout=15).content)).convert("RGB")


def draw_detections(ax, image, detections, title=""):
    """Draw bounding boxes and segmentation masks on axes."""
    W, H = image.size
    ax.imshow(image)

    for i, det in enumerate(detections):
        color = COLORS[i % len(COLORS)]
        xy, hw = det["xy"], det["hw"]

        # Draw segmentation mask
        if "mask" in det:
            mask_arr = det["mask"]
            if isinstance(mask_arr, mx.array):
                mask_arr = np.array(mask_arr)
            if mask_arr.shape != (H, W):
                mask_arr = np.array(
                    Image.fromarray(mask_arr.astype(np.uint8)).resize((W, H), Image.NEAREST)
                ).astype(bool)
            overlay = np.zeros((H, W, 4))
            for c in range(3):
                overlay[:, :, c] = (mask_arr > 0) * color[c]
            overlay[:, :, 3] = (mask_arr > 0) * 0.5
            ax.imshow(overlay)

        # Draw bounding box
        cx, cy = xy["x"] * W, xy["y"] * H
        bw, bh = hw["w"] * W, hw["h"] * H
        x0, y0 = cx - bw / 2, cy - bh / 2

        rect = patches.Rectangle(
            (x0, y0), bw, bh,
            linewidth=2.5, edgecolor=color, facecolor="none",
        )
        ax.add_patch(rect)
        ax.text(
            x0, max(y0 - 8, 12), f"det {i}",
            color="white", fontsize=11, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor=color, alpha=0.85),
        )

    ax.set_title(title, fontsize=14)
    ax.axis("off")

## Load Model

In [ ]:
from mlx_vlm import load

model, processor = load("tiiuae/Falcon-Perception")

In [ ]:
# Load test image
image = fetch_image("http://images.cocodataset.org/val2017/000000039769.jpg")
image

## 1. Object Detection

Detect objects by text query — returns bounding boxes, sizes, and segmentation masks.

In [ ]:
detections = model.generate_perception(
    processor,
    image="http://images.cocodataset.org/val2017/000000039769.jpg",
    query="cats",
    max_new_tokens=512,
)

print(f'Query: "cat" — {len(detections)} detection(s)')
for i, det in enumerate(detections):
    xy, hw = det["xy"], det["hw"]
    has_mask = "mask" in det
    print(f"  [{i}] Center: ({xy['x']:.3f}, {xy['y']:.3f}), Size: ({hw['h']:.3f}, {hw['w']:.3f}), Mask: {has_mask}")

## 2. Visualize Detections

Plot bounding boxes and segmentation masks overlaid on the image.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
draw_detections(ax, image, detections, title='Detection + Segmentation — "cat"')
plt.tight_layout()
plt.show()

## 3. Mask Overlay on Image

Blend segmentation masks directly onto the image pixels.

In [ ]:
W, H = image.size
overlay = np.array(image).copy()
colors_rgb = [(30, 120, 255), (255, 80, 30), (30, 200, 30)]

for i, det in enumerate(detections):
    if "mask" not in det:
        continue
    mask = det["mask"]
    if isinstance(mask, mx.array):
        mask = np.array(mask)
    if mask.shape != (H, W):
        mask = np.array(Image.fromarray(mask.astype(np.float32)).resize((W, H)))
    binary = mask > 0
    color = np.array(colors_rgb[i % len(colors_rgb)])
    overlay[binary] = (overlay[binary] * 0.5 + color * 0.5).astype(np.uint8)

Image.fromarray(overlay)

## 4. Multi-Query Detection

Run different text queries on different images.

In [ ]:
urls_queries = [
    ("http://images.cocodataset.org/val2017/000000039769.jpg", "cats"),
    ("http://images.cocodataset.org/val2017/000000397133.jpg", "airplanes"),
    ("http://images.cocodataset.org/val2017/000000252219.jpg", "zebras"),
    ("http://images.cocodataset.org/val2017/000000087038.jpg", "cars"),
]

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()

for idx, (url, query) in enumerate(urls_queries):
    print(f'[{idx+1}/4] "{query}"...', end=" ", flush=True)
    try:
        img = fetch_image(url)
    except Exception as e:
        print(f"FAILED ({e})")
        axes[idx].text(0.5, 0.5, "Failed to load", ha="center", va="center", fontsize=14)
        axes[idx].set_title(f'"{query}"')
        axes[idx].axis("off")
        continue

    dets = model.generate_perception(
        processor,
        image=url,
        query=query,
        max_new_tokens=512,
    )
    print(f"{len(dets)} detection(s)")
    draw_detections(axes[idx], img, dets, title=f'"{query}" ({len(dets)} det.)')

plt.suptitle("Falcon-Perception MLX — Multi-Query Detection", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()